# Notebook 04: Deploying RAG Chatbot to AWS (ECR + ECS Fargate)

This notebook guides you through deploying the Streamlit chatbot application to AWS using:
- **Amazon ECR** (Elastic Container Registry) - for storing Docker images
- **Amazon ECS** (Elastic Container Service) - for running containers
- **AWS Fargate** - serverless container compute

## Prerequisites

Before starting:
- AWS CLI configured with appropriate permissions
- Docker installed (or use SageMaker Notebook Instance)
- Your `.env` file with valid credentials
- Completed notebooks 00-03

---

## Part 1: Create ECR Repository

ECR is AWS's Docker container registry, similar to Docker Hub.

### Option A: Via AWS Console (Easiest)

1. Go to **ECR Console**: https://console.aws.amazon.com/ecr/
2. Click **Get Started** or **Create repository**
3. Configure:
   - **Visibility**: Private
   - **Repository name**: `ecommerce-rag-chatbot`
   - **Tag immutability**: Disabled (default)
4. Click **Create repository**
5. **Copy the Repository URI** (looks like: `123456789012.dkr.ecr.us-east-1.amazonaws.com/ecommerce-rag-chatbot`)

### Option B: Via AWS CLI

Run this command in your terminal:

In [ ]:
# Create ECR repository via CLI
!aws ecr create-repository --repository-name ecommerce-rag-chatbot --region us-east-1

**Expected output:**
```json
{
    "repository": {
        "repositoryArn": "arn:aws:ecr:us-east-1:123456789012:repository/ecommerce-rag-chatbot",
        "registryId": "123456789012",
        "repositoryName": "ecommerce-rag-chatbot",
        "repositoryUri": "123456789012.dkr.ecr.us-east-1.amazonaws.com/ecommerce-rag-chatbot"
    }
}
```

---

## Part 2: Build and Push Docker Image

### Environment Setup

First, set your AWS account ID and region:

In [ ]:
# Get AWS Account ID
!export AWS_ACCOUNT_ID=$(aws sts get-caller-identity --query Account --output text)
!export AWS_REGION=us-east-1
!echo "Account ID: $AWS_ACCOUNT_ID"
!echo "Region: $AWS_REGION"

### IMPORTANT: SageMaker Role Permissions

If you're building and pushing from **SageMaker Notebook Instance**, the SageMaker execution role needs these permissions:

#### Required IAM Policies:

1. **`AmazonEC2ContainerRegistryPowerUser`** - For ECR push access
2. **`AmazonECS_FullAccess`** - For ECS cluster/service management

#### How to Add Permissions:

1. Go to **IAM Console** → **Roles**
2. Find your SageMaker execution role (e.g., `AmazonSageMakerAdminIAMExecutionRole`)
3. Click **Add permissions** → **Attach policies**
4. Search for and attach both policies above
5. Click **Attach policies**

**Common Errors Without Permissions:**
- `denied: User is not authorized to perform: ecr:InitiateLayerUpload`
- `AccessDeniedException: not authorized to perform: ecs:CreateCluster`

Make sure to add these permissions **before** running the commands below.

### Step 1: Login to ECR

In [ ]:
# Login to ECR (update account ID and region as needed)
!aws ecr get-login-password --region us-east-1 | \
    docker login --username AWS --password-stdin 123456789012.dkr.ecr.us-east-1.amazonaws.com

**Expected output:** `Login Succeeded`

### Step 2: Build Docker Image

Make sure you're in the project root directory (where `Dockerfile` is located):

In [ ]:
# Navigate to project root (adjust path as needed)
%cd /Users/apalakkode/Library/CloudStorage/OneDrive-PayPal/Agentic\ RAG/ecommerce-chatbot/

# Verify Dockerfile exists
!ls -la Dockerfile

In [ ]:
# Build Docker image (this will take 5-10 minutes)
# Use --no-cache to ensure .env changes are picked up
!docker build --no-cache -t ecommerce-rag-chatbot .

**What happens during build:**
- Pulls base Python 3.9 image
- Installs system dependencies (build-essential, curl)
- Installs Python packages from `requirements-app.txt`
- Copies application code (`src/`, `data/`, `app.py`)
- **Copies `.env` file** (contains your API keys)
- Sets up Streamlit to run on port 8501

### Step 3: Tag the Image

In [ ]:
# Tag the image for ECR (replace with your account ID)
!docker tag ecommerce-rag-chatbot:latest \
    123456789012.dkr.ecr.us-east-1.amazonaws.com/ecommerce-rag-chatbot:latest

### Step 4: Push to ECR

In [ ]:
# Push to ECR (this will take 5-10 minutes depending on network speed)
!docker push 123456789012.dkr.ecr.us-east-1.amazonaws.com/ecommerce-rag-chatbot:latest

**Expected output:**
```
The push refers to repository [123456789012.dkr.ecr.us-east-1.amazonaws.com/ecommerce-rag-chatbot]
...
latest: digest: sha256:abc123... size: 4567
```

Image is now stored in ECR and ready for deployment!

---

## Part 3: Create ECS Cluster

ECS (Elastic Container Service) orchestrates your Docker containers.

### Option A: Via AWS Console (Recommended)

1. Go to **ECS Console**: https://console.aws.amazon.com/ecs/
2. Click **Clusters** (left sidebar) → **Create cluster**
3. Configure:
   - **Cluster name**: `ecommerce-rag-cluster` (or use default)
   - **Infrastructure**: AWS Fargate (serverless)
   - **Monitoring**: CloudWatch Container Insights (optional, costs extra)
4. Click **Create**

### Option B: Via AWS CLI

In [ ]:
# Create ECS cluster via CLI
!aws ecs create-cluster \
    --cluster-name ecommerce-rag-cluster \
    --capacity-providers FARGATE FARGATE_SPOT \
    --region us-east-1

**Note:** If you get `AccessDeniedException`, make sure your IAM role has `AmazonECS_FullAccess` policy attached.

---

## Part 4: Create Task Definition

A Task Definition is like a blueprint for your container - it defines CPU, memory, image, ports, etc.

### Via AWS Console (Recommended)

1. In **ECS Console**, click **Task Definitions** → **Create new task definition**

2. **Infrastructure requirements:**
   - **Launch type**: AWS Fargate
   - **Operating system/Architecture**: Linux/X86_64
   - **CPU**: 0.5 vCPU (or 1 vCPU for better performance)
   - **Memory**: 1 GB (or 2 GB for better performance)
   - **Task role**: Leave blank or use `ecsTaskExecutionRole`
   - **Task execution role**: `ecsTaskExecutionRole` (auto-created by AWS)

3. **Container - 1:**
   - **Container name**: `ecommerce-rag-app`
   - **Image URI**: `123456789012.dkr.ecr.us-east-1.amazonaws.com/ecommerce-rag-chatbot:latest`
   - **Essential container**: Yes (checked)
   - **Port mappings**:
     - Container port: **8501**
     - Protocol: **TCP**
     - App protocol: **HTTP**
   - **Environment variables**: None needed (they're in the Docker image via `.env`)

4. **Task definition family name**: `ecommerce-rag-task`

5. Click **Create**

### Performance Notes:

| Configuration | Cost/Hour | Performance | Recommended For |
|--------------|-----------|-------------|------------------|
| 0.5 vCPU, 1GB | ~$0.04 | Slow responses (5-10s) | Development/testing |
| 1 vCPU, 2GB | ~$0.08 | Faster responses (3-5s) | **Production/demos** |
| 2 vCPU, 4GB | ~$0.16 | Fast responses (2-3s) | High-traffic production |

---

## Part 5: Create and Deploy Service

A Service ensures your task keeps running and handles networking.

### Via AWS Console

1. Go to your cluster → **Services** tab → **Create**

2. **Environment:**
   - **Compute options**: Launch type
   - **Launch type**: FARGATE

3. **Deployment configuration:**
   - **Application type**: Service
   - **Family**: Select `ecommerce-rag-task`
   - **Service name**: `ecommerce-rag-service`
   - **Desired tasks**: 1

4. **Networking:**
   - **VPC**: Select your default VPC
   - **Subnets**: Select all available **public subnets** (check 2-3 boxes)
   - **Security group**: Create new
     - **Security group name**: `ecommerce-rag-sg`
     - **Inbound rules**:
       - Type: **Custom TCP**
       - **Port range**: **8501** (CRITICAL - don't forget!)
       - Source: **Anywhere-IPv4 (0.0.0.0/0)**
   - **Public IP**: **ENABLED** ✅ (CRITICAL - must be turned ON!)

5. **Load balancing**: None

6. Click **Create**

### Common Mistakes to Avoid:

- ❌ Forgetting to specify port **8501** in security group → Connection timeout
- ❌ Public IP turned OFF → Can't access from internet
- ❌ Using private subnets instead of public → No internet access
- ❌ Using 0.5 vCPU without warning users → Very slow responses

---

## Part 6: Access Your Deployed Application

### Step 1: Wait for Task to Start

1. In your cluster, go to **Tasks** tab
2. Wait for **Last status: RUNNING** (takes 2-3 minutes)
3. If status shows **STOPPED**, click on it and check **Logs** tab for errors

### Step 2: Get Public IP

1. Click on the running **Task ID**
2. In **Configuration** section, find **Public IP**
3. Copy the IP address (e.g., `3.238.85.28`)

### Step 3: Open in Browser

Open: `http://<public-ip>:8501`

Example: `http://3.238.85.28:8501`

**Note:** Use `http://` not `https://`

---

## Troubleshooting

### Problem: Connection Timeout (ERR_CONNECTION_TIMED_OUT)

**Cause:** Security group doesn't allow port 8501

**Fix:**
1. Go to task → Configuration → Network → Click security group ID
2. Edit inbound rules
3. Add rule: Custom TCP, Port 8501, Source 0.0.0.0/0

### Problem: Task Keeps Stopping

**Cause:** Application error or missing dependencies

**Fix:**
1. Click on stopped task → **Logs** tab
2. Look for error messages
3. Common issues:
   - `.env` file missing → Rebuild Docker image
   - Port already in use → Not possible in Fargate
   - Out of memory → Increase task memory to 2GB

### Problem: Permission Denied Errors During Push

**Cause:** IAM role lacks ECR permissions

**Fix:**
1. Go to IAM → Roles → Find SageMaker execution role
2. Attach policy: `AmazonEC2ContainerRegistryPowerUser`
3. Wait 30 seconds, retry

### Problem: Page Loads But Chatbot Doesn't Respond

**Cause:** API keys in `.env` are invalid or missing

**Fix:**
1. Update `.env` file locally
2. Rebuild Docker image: `docker build --no-cache -t ecommerce-rag-chatbot .`
3. Re-tag and push to ECR
4. Update ECS service to force new deployment:
   ```bash
   aws ecs update-service \
       --cluster ecommerce-rag-cluster \
       --service ecommerce-rag-service \
       --force-new-deployment \
       --region us-east-1
   ```

---

## Updating Your Application

When you make code changes:

### Step 1: Rebuild and Push

In [ ]:
# Rebuild Docker image
!docker build --no-cache -t ecommerce-rag-chatbot .

# Tag and push
!docker tag ecommerce-rag-chatbot:latest \
    123456789012.dkr.ecr.us-east-1.amazonaws.com/ecommerce-rag-chatbot:latest
!docker push 123456789012.dkr.ecr.us-east-1.amazonaws.com/ecommerce-rag-chatbot:latest

### Step 2: Force New Deployment

In [ ]:
# Force ECS to pull the new image and redeploy
!aws ecs update-service \
    --cluster ecommerce-rag-cluster \
    --service ecommerce-rag-service \
    --force-new-deployment \
    --region us-east-1

Wait 2-3 minutes for the new task to start, then refresh your browser.

---

## Cleanup (To Avoid Charges)

When done with the demo:

### Option 1: Stop the Service (Quick, Can Resume Later)

In [ ]:
# Set desired tasks to 0 (stops running containers but keeps configuration)
!aws ecs update-service \
    --cluster ecommerce-rag-cluster \
    --service ecommerce-rag-service \
    --desired-count 0 \
    --region us-east-1

### Option 2: Delete Everything (Complete Cleanup)

**Via Console:**
1. Delete ECS Service (set desired count to 0 first, wait, then delete)
2. Delete ECS Cluster
3. Delete ECR Repository (in ECR console)
4. Delete CloudWatch Log Group: `/ecs/ecommerce-rag-task`

**Via CLI:**

In [ ]:
# Delete service
!aws ecs delete-service \
    --cluster ecommerce-rag-cluster \
    --service ecommerce-rag-service \
    --force \
    --region us-east-1

# Delete cluster
!aws ecs delete-cluster \
    --cluster ecommerce-rag-cluster \
    --region us-east-1

# Delete ECR repository
!aws ecr delete-repository \
    --repository-name ecommerce-rag-chatbot \
    --force \
    --region us-east-1

---

## Cost Estimation

### Fargate Costs (us-east-1):

| Configuration | Per Hour | Per Day (24h) | Per Month |
|--------------|----------|---------------|------------|
| 0.5 vCPU, 1GB | $0.04 | $0.96 | ~$29 |
| 1 vCPU, 2GB | $0.08 | $1.92 | ~$58 |
| 2 vCPU, 4GB | $0.16 | $3.84 | ~$115 |

### Additional Costs:
- **ECR storage**: $0.10/GB/month (image is ~500MB = $0.05/month)
- **Data transfer**: First 100GB/month free
- **CloudWatch Logs**: ~$0.50-1/month for typical usage

### Cost-Saving Tips:
1. **Stop when not in use**: Set desired tasks to 0 after demos
2. **Use Fargate Spot**: Save ~70% but tasks can be interrupted
3. **Right-size resources**: Don't use 2 vCPU if 1 vCPU is enough
4. **Demo locally**: Use `streamlit run app.py` for development

---

## Summary

You've successfully deployed your RAG chatbot to AWS!

**What you accomplished:**
-  Created ECR repository and pushed Docker image
-  Set up ECS Fargate cluster (serverless containers)
-  Configured networking (VPC, subnets, security groups)
-  Deployed the Streamlit application with public access
-  Learned troubleshooting and cost optimization

**Architecture:**
```
User Browser → Internet → Public IP:8501 → 
    ECS Fargate Task (Streamlit) → 
        AWS Bedrock Knowledge Base (retrieval) → 
        OpenAI API (LLM generation) → 
        Response back to user
```

**Next steps:**
- Share the public URL with stakeholders
- Monitor CloudWatch logs for usage patterns
- Consider adding Application Load Balancer for custom domain
- Implement authentication (e.g., Streamlit auth or AWS Cognito)
- Scale to multiple tasks for high availability